# DSA Week 1 -- Big-O & Python Cost Model

**Course:** Data Structures & Algorithms (Year 2, Semester 4)
**Session:** 3 hours
**Prerequisites:** CP1 + CP2 + OOP
**Focus:** Big-O notation, list/dict/set costs

## Learning Objectives

By the end of this session you will be able to:

1. Explain what Big-O notation means in plain English
2. Classify common operations as O(1), O(log n), O(n), O(n log n), or O(n^2)
3. Predict the cost of Python list, dict, and set operations
4. Count operations in a simple algorithm and derive its Big-O
5. Visualize growth rates and understand why they matter at scale
6. Identify the bottleneck in a piece of code

## Why This Week Matters

You already know how to write Python code that *works*. This semester is about
making code that works **fast**. In CP1 you processed 8 rows of data. In the
real world your pipeline will handle 100,000 or 1,000,000 rows. The difference
between O(n) and O(n^2) at that scale is the difference between "done in 1 second"
and "still running after 3 hours."

Big-O is the language engineers use to talk about performance. After today you
will never look at a loop the same way again.

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/ArifSolmaz/courseos-curriculum.git
# %cd courseos-curriculum/course-content

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: What is Big-O? (Plain English First)

Big-O describes **how an algorithm's time grows** as the input gets bigger.
It answers one question: *"If I have 10x more data, how much longer does it take?"*

### The Restaurant Analogy

Imagine you run a restaurant:

```
Operation                  | Big-O     | Restaurant Analogy
---------------------------|-----------|--------------------------------------------
Look up reservation by ID  | O(1)      | The host checks a numbered list -- instant
Find someone by name       | O(n)      | Walk through every table asking names
Seat everyone optimally    | O(n log n)| Sort all guests by party size, then seat
Compare every pair         | O(n^2)    | Every guest shakes hands with every other
Try every seating combo    | O(2^n)    | Try EVERY possible arrangement -- impossible
```

### The Key Insight

Big-O drops constants and lower-order terms because at large scale,
only the **dominant term** matters:

```
Actual steps: 3n^2 + 5n + 100
Big-O:        O(n^2)

Why? At n = 1,000,000:
  3n^2  = 3,000,000,000,000   (3 trillion -- dominates!)
  5n    = 5,000,000            (5 million -- rounding error)
  100   = 100                  (irrelevant)
```

Think of it like measuring distance between cities: you say "300 km"
not "300 km, 47 meters, and 12 centimeters." The small parts do not matter.

### Example 1: O(1) vs O(n) -- The Speed Gap

Let us see the difference between constant-time and linear-time lookup.
This is the single most important optimization you will learn.

In [ ]:
import time

# Build test data: 1 million items
n = 1_000_000
data_dict = {i: "value_" + str(i) for i in range(n)}
data_list = list(range(n))
data_set = set(range(n))

target = n - 1  # worst case for list

# O(1) -- dict lookup
start = time.time()
result = data_dict[target]
t_dict = time.time() - start

# O(n) -- list search
start = time.time()
found = target in data_list
t_list = time.time() - start

# O(1) -- set membership
start = time.time()
found = target in data_set
t_set = time.time() - start

print("=== Lookup Speed Comparison (n = 1,000,000) ===")
print()
print("  dict[key]     : " + "{:.1f}".format(t_dict * 1e6) + " microseconds  -- O(1)")
print("  item in list  : " + "{:.1f}".format(t_list * 1000) + " milliseconds  -- O(n)")
print("  item in set   : " + "{:.1f}".format(t_set * 1e6) + " microseconds  -- O(1)")
print()
if t_list > 0 and t_dict > 0:
    print("  dict is ~" + str(int(t_list / t_dict)) + "x faster than list!")
    print("  This is why choosing the right data structure matters.")

**Expected Output** (approximate):
```
=== Lookup Speed Comparison (n = 1,000,000) ===

  dict[key]     : 0.5 microseconds  -- O(1)
  item in list  : 12.3 milliseconds  -- O(n)
  item in set   : 0.3 microseconds  -- O(1)

  dict is ~24000x faster than list!
  This is why choosing the right data structure matters.
```

**What just happened:**
- `dict[key]` computes a hash, jumps directly to the slot -- O(1)
- `item in list` must scan from index 0 to 999,999 -- O(n)
- `item in set` uses the same hash trick as dict -- O(1)

> **Real-world impact:** If your pipeline checks membership 10,000 times
> on a list of 1M items, switching to a set saves you from 10,000 x 12ms
> = 2 minutes down to 10,000 x 0.5us = 5 milliseconds.

---
### Example 2: Python Data Structure Cost Card

This is your **cheat sheet** for the entire semester. Bookmark this cell.
Every time you write code, ask: "What is the Big-O of this operation?" 

In [ ]:
print("""
============================================================
         PYTHON DATA STRUCTURE COST CARD
============================================================

LIST (array-based):
  Access by index   lst[i]        -> O(1)
  Append            lst.append(x) -> O(1) amortized
  Pop from end      lst.pop()     -> O(1)
  Pop from front    lst.pop(0)    -> O(n)  *** SLOW ***
  Insert at front   lst.insert(0) -> O(n)  *** SLOW ***
  Search            x in lst      -> O(n)
  Sort              lst.sort()    -> O(n log n)
  Slice             lst[a:b]      -> O(b - a)
  Length             len(lst)      -> O(1)

DICT (hash table):
  Lookup by key     d[k]          -> O(1) average
  Insert            d[k] = v      -> O(1) average
  Delete            del d[k]      -> O(1) average
  Membership        k in d        -> O(1) average
  Iterate keys      for k in d    -> O(n)

SET (hash set):
  Membership        x in s        -> O(1) average
  Add               s.add(x)      -> O(1) average
  Remove            s.remove(x)   -> O(1) average
  Union             s | t         -> O(len(s) + len(t))
  Intersection      s & t         -> O(min(len(s), len(t)))

TUPLE: same as list for access, but immutable (no append/insert)

DEQUE (collections.deque):
  Append right      dq.append(x)     -> O(1)
  Append left       dq.appendleft(x) -> O(1)
  Pop right         dq.pop()         -> O(1)
  Pop left          dq.popleft()     -> O(1)
  Access by index   dq[i]            -> O(n)  *** SLOW ***

============================================================
RULE: use dict/set for lookups, list for ordered sequences,
      deque for queue operations (add/remove from both ends).
============================================================
""")

**Key Takeaway:** The three operations that trap beginners are:
1. `x in list` -- O(n). Use `x in set` or `x in dict` instead -- O(1)
2. `list.pop(0)` or `list.insert(0, x)` -- O(n). Use `collections.deque` instead -- O(1)
3. Nested loops over a list for matching -- O(n^2). Build a dict first, then loop once -- O(n)

---
### Example 3: Counting Operations -- Learning to See Big-O

The best way to understand Big-O is to **count** how many operations
an algorithm performs at different input sizes. Let us do that explicitly.

In [ ]:
def linear_search_counted(data, target):
    """O(n) -- check every element, count operations."""
    ops = 0
    for item in data:
        ops += 1
        if item == target:
            return ops, True
    return ops, False

def binary_search_counted(sorted_data, target):
    """O(log n) -- halve the search space, count operations."""
    ops = 0
    low, high = 0, len(sorted_data) - 1
    while low <= high:
        ops += 1
        mid = (low + high) // 2
        if sorted_data[mid] == target:
            return ops, True
        elif sorted_data[mid] < target:
            low = mid + 1
        else:
            high = mid - 1
    return ops, False

def nested_loop_counted(data):
    """O(n^2) -- compare every pair, count operations."""
    ops = 0
    n = len(data)
    for i in range(n):
        for j in range(i + 1, n):
            ops += 1
    return ops

# Compare at different sizes
print("=== Operation Counts at Different Input Sizes ===")
print()
print("  n         | Linear O(n) | Binary O(log n) | Pairs O(n^2)")
print("  ----------|-------------|-----------------|-------------")
for n in [10, 100, 1_000, 10_000, 100_000]:
    data = list(range(n))
    target = n - 1  # worst case

    lin_ops, _ = linear_search_counted(data, target)
    bin_ops, _ = binary_search_counted(data, target)
    # Only compute pairs for small n (otherwise too slow)
    if n <= 10_000:
        pair_ops = nested_loop_counted(data)
    else:
        pair_ops = n * (n - 1) // 2  # formula

    print("  " + "{:>9,}".format(n) + " | " + "{:>11,}".format(lin_ops) + " | " + "{:>15,}".format(bin_ops) + " | " + "{:>11,}".format(pair_ops))

print()
print("Key insight: at n = 100,000:")
print("  Linear does    100,000 operations")
print("  Binary does         17 operations")
print("  Pairs does  5,000,000,000 operations (5 BILLION)")

**Expected Output:**
```
=== Operation Counts at Different Input Sizes ===

  n         | Linear O(n) | Binary O(log n) | Pairs O(n^2)
  ----------|-------------|-----------------|-------------
         10 |          10 |               4 |          45
        100 |         100 |               7 |       4,950
      1,000 |       1,000 |              10 |     499,500
     10,000 |      10,000 |              14 |  49,995,000
    100,000 |     100,000 |              17 | 4,999,950,000

Key insight: at n = 100,000:
  Linear does    100,000 operations
  Binary does         17 operations
  Pairs does  5,000,000,000 operations (5 BILLION)
```

**The 10x test:**
- O(n): 10x more data = 10x more time (linear growth)
- O(log n): 10x more data = ~3 more steps (barely grows!)
- O(n^2): 10x more data = 100x more time (explosive growth!)

### Try It Yourself #1

What is the Big-O of each code snippet below? Add your answer as a comment.

In [ ]:
# Snippet A: What is the Big-O?
def snippet_a(data):
    return data[0] + data[-1]
# Answer: O(__)

# Snippet B: What is the Big-O?
def snippet_b(data):
    total = 0
    for x in data:
        total += x
    return total
# Answer: O(__)

# Snippet C: What is the Big-O?
def snippet_c(data):
    for i in range(len(data)):
        for j in range(len(data)):
            if data[i] == data[j] and i != j:
                return True
    return False
# Answer: O(__)

# Snippet D: What is the Big-O?
def snippet_d(data):
    return len(data)
# Answer: O(__)

# Snippet E: What is the Big-O?
def snippet_e(data):
    s = set(data)       # step 1
    return 42 in s      # step 2
# Answer for step 1: O(__)
# Answer for step 2: O(__)
# Answer combined:   O(__)

---
## Part 2: Visualizing Growth Rates

Numbers are one thing -- seeing the curves is another. This plot will make
Big-O intuitive for you forever.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import math
import os

ns = list(range(1, 101))
o1     = [1] * len(ns)
olog   = [math.log2(n) if n > 0 else 0 for n in ns]
on     = list(ns)
onlogn = [n * math.log2(n) if n > 0 else 0 for n in ns]
on2    = [n ** 2 for n in ns]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: all curves, limited y-axis
ax = axes[0]
ax.plot(ns, o1, label="O(1)", linewidth=2)
ax.plot(ns, olog, label="O(log n)", linewidth=2)
ax.plot(ns, on, label="O(n)", linewidth=2)
ax.plot(ns, onlogn, label="O(n log n)", linewidth=2)
ax.plot(ns, on2, label="O(n^2)", linewidth=2, linestyle="--")
ax.set_title("Growth Rates (zoomed to 500)", fontsize=13)
ax.set_xlabel("Input Size (n)")
ax.set_ylabel("Operations")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 500)

# Right: log scale to see the full picture
ax2 = axes[1]
ax2.plot(ns, o1, label="O(1)", linewidth=2)
ax2.plot(ns, olog, label="O(log n)", linewidth=2)
ax2.plot(ns, on, label="O(n)", linewidth=2)
ax2.plot(ns, onlogn, label="O(n log n)", linewidth=2)
ax2.plot(ns, on2, label="O(n^2)", linewidth=2, linestyle="--")
ax2.set_title("Growth Rates (log scale)", fontsize=13)
ax2.set_xlabel("Input Size (n)")
ax2.set_ylabel("Operations (log scale)")
ax2.set_yscale("log")
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs("reports/benchmark", exist_ok=True)
fig.savefig("reports/benchmark/growth_rates.png", dpi=100, bbox_inches="tight")
plt.close(fig)
print("Saved: reports/benchmark/growth_rates.png")
print()
print("Notice how O(n^2) explodes upward while O(log n) barely rises.")
print("At n=100: O(n^2)=10,000 but O(log n)=7. That is 1,400x difference!")

**Expected Output:**
```
Saved: reports/benchmark/growth_rates.png

Notice how O(n^2) explodes upward while O(log n) barely rises.
At n=100: O(n^2)=10,000 but O(log n)=7. That is 1,400x difference!
```

---
## Part 3: Proving the Cost Model with Experiments

Theory is nice, but engineers **measure**. Let us verify the cost card
with actual timing experiments.

In [ ]:
import time

def time_operation(label, func, repeat=5):
    """Time a function, return average in microseconds."""
    times = []
    for _ in range(repeat):
        start = time.time()
        func()
        times.append(time.time() - start)
    avg_us = sum(times) / len(times) * 1e6
    print("  " + label.ljust(35) + ": " + "{:>10.1f}".format(avg_us) + " us")
    return avg_us

n = 100_000
test_list = list(range(n))
test_dict = {i: i for i in range(n)}
test_set = set(range(n))

print("=== Timing Python Operations (n = " + "{:,}".format(n) + ") ===")
print()

print("LIST operations:")
time_operation("list[0] (front access)", lambda: test_list[0])
time_operation("list[-1] (back access)", lambda: test_list[-1])
time_operation("list[n//2] (middle access)", lambda: test_list[n // 2])
time_operation("99999 in list (worst case)", lambda: 99999 in test_list)
time_operation("0 in list (best case)", lambda: 0 in test_list)
print()

print("DICT operations:")
time_operation("dict[99999] (lookup)", lambda: test_dict[99999])
time_operation("99999 in dict (membership)", lambda: 99999 in test_dict)
print()

print("SET operations:")
time_operation("99999 in set (membership)", lambda: 99999 in test_set)
print()

print("CONCLUSION:")
print("  Index access (list[i]) and dict/set lookups are all O(1) -- microseconds.")
print("  Searching inside a list (x in list) is O(n) -- much slower!")

**Expected Output** (times vary by machine):
```
=== Timing Python Operations (n = 100,000) ===

LIST operations:
  list[0] (front access)            :        0.2 us
  list[-1] (back access)            :        0.2 us
  list[n//2] (middle access)        :        0.2 us
  99999 in list (worst case)        :     1200.0 us
  0 in list (best case)             :        0.2 us

DICT operations:
  dict[99999] (lookup)              :        0.3 us
  99999 in dict (membership)        :        0.2 us

SET operations:
  99999 in set (membership)         :        0.2 us

CONCLUSION:
  Index access (list[i]) and dict/set lookups are all O(1) -- microseconds.
  Searching inside a list (x in list) is O(n) -- much slower!
```

---
## Part 4: The Three Performance Traps

These are the most common mistakes that make student code slow.
After this semester you will never make them again.

### Trap 1: Membership testing on a list

In [ ]:
import time

# BAD: checking membership in a list inside a loop
def find_duplicates_bad(data):
    """O(n^2) -- for each item, scan the seen list."""
    seen = []  # <-- THIS IS THE PROBLEM
    duplicates = []
    for item in data:
        if item in seen:        # O(n) scan every time!
            duplicates.append(item)
        seen.append(item)
    return duplicates

# GOOD: checking membership in a set inside a loop
def find_duplicates_good(data):
    """O(n) -- for each item, check the seen set."""
    seen = set()  # <-- O(1) membership test
    duplicates = []
    for item in data:
        if item in seen:        # O(1) check!
            duplicates.append(item)
        seen.add(item)
    return duplicates

# Test correctness
import random
random.seed(42)
data = [random.randint(0, 5000) for _ in range(10_000)]

d1 = find_duplicates_bad(data)
d2 = find_duplicates_good(data)
assert sorted(d1) == sorted(d2), "Both must find same duplicates"
print("Both find " + str(len(d1)) + " duplicates -- correctness verified!")
print()

# Time them
start = time.time()
find_duplicates_bad(data)
t_bad = time.time() - start

start = time.time()
find_duplicates_good(data)
t_good = time.time() - start

print("BAD  (list): " + "{:.4f}".format(t_bad) + "s -- O(n^2)")
print("GOOD (set):  " + "{:.4f}".format(t_good) + "s -- O(n)")
print("Speedup:     " + str(int(t_bad / t_good)) + "x")

**Expected Output:**
```
Both find 4877 duplicates -- correctness verified!

BAD  (list): 1.2345s -- O(n^2)
GOOD (set):  0.0012s -- O(n)
Speedup:     1000x
```

### Trap 2: Building strings with += in a loop

```python
# BAD: O(n^2) -- each += creates a new string
result = ""
for word in words:
    result += word + " "    # copies entire string each time!

# GOOD: O(n) -- join does one allocation
result = " ".join(words)
```

### Trap 3: Using list.pop(0) or list.insert(0, x)

```python
# BAD: O(n) per operation -- shifts all elements
queue = [1, 2, 3, 4, 5]
item = queue.pop(0)         # shifts 4 elements left

# GOOD: O(1) per operation
from collections import deque
queue = deque([1, 2, 3, 4, 5])
item = queue.popleft()      # no shifting needed
```

### Try It Yourself #2

Fix this slow function. It finds all numbers that appear in BOTH lists.
The current version is O(n*m). Make it O(n+m).

In [ ]:
# SLOW VERSION -- O(n * m)
def common_elements_slow(list_a, list_b):
    result = []
    for item in list_a:
        if item in list_b:  # O(m) scan each time!
            result.append(item)
    return result

# TODO: Write a FAST version -- O(n + m)
def common_elements_fast(list_a, list_b):
    # Hint: convert one list to a set first
    pass

# Test
a = list(range(0, 10000, 2))    # even numbers
b = list(range(0, 10000, 3))    # multiples of 3
# Expected: numbers divisible by both 2 and 3 (i.e., multiples of 6)

result = common_elements_slow(a, b)
print("Common elements:", len(result))
# TODO: time both versions and print the speedup

---
## Part 5: Why Big-O Matters for Your Project

Every project track has performance bottlenecks that Big-O thinking can solve:

| Track | Product | Typical Bottleneck | Fix |
|-------|---------|-------------------|-----|
| Robotics | MechaSense Studio | Scanning all sensor readings for anomalies | Hash index on sensor ID |
| Data/AI | CleanReport Pipeline | Deduplicating records with nested loops | Set-based dedup |
| Simulation | SimLab Engine | Checking every pair of objects for collision | Spatial hash grid |
| Space | Lightcurve Explorer | Searching sorted time-series for transit events | Binary search |
| IoT | AutoDashboard Reporter | Aggregating across thousands of devices | Dict-based grouping |

This semester you will implement at least one of these optimizations and **prove**
it is faster with benchmarks. Today's lesson gives you the vocabulary to analyze
what is slow and predict what will help.

In [ ]:
# Quick demo: the "before and after" pattern you will use all semester
import time

# Simulate a pipeline bottleneck: finding all readings above threshold
n = 500_000
readings = list(range(n))

# BEFORE: scan all readings looking for matches to a target set
targets = list(range(0, n, 100))  # every 100th value

start = time.time()
# BAD: nested loop
matches_slow = [r for r in readings if r in targets]
t_slow = time.time() - start

# AFTER: convert targets to a set
target_set = set(targets)
start = time.time()
matches_fast = [r for r in readings if r in target_set]
t_fast = time.time() - start

print("Results match:", len(matches_slow) == len(matches_fast))
print("Before (list targets): " + "{:.3f}".format(t_slow) + "s")
print("After  (set targets):  " + "{:.3f}".format(t_fast) + "s")
print("Speedup: " + "{:.0f}".format(t_slow / t_fast) + "x")
print()
print("This is EXACTLY the kind of optimization you will do in your project.")

**Expected Output:**
```
Results match: True
Before (list targets): 5.123s
After  (set targets):  0.045s
Speedup: 114x

This is EXACTLY the kind of optimization you will do in your project.
```

---
## Common Mistakes with Big-O

| Mistake | Why It Is Wrong | Correct |
|---------|----------------|---------|
| "O(2n) because of two loops" | Constants are dropped | O(n) -- two sequential loops are still O(n) |
| "dict lookup is O(n)" | Dict uses hash table | O(1) average |
| "sorted() is O(n)" | Sorting requires comparisons | O(n log n) |
| "O(1) because it is one line" | One line can hide a loop | `x in list` is O(n) even though it is one line |
| "O(n^2) because nested loops" | Not always! | Only if BOTH loops go to n. `for i in range(n): for j in range(5)` is O(n) |

### Quick Rules for Spotting Big-O

1. **No loops:** probably O(1)
2. **One loop over n items:** O(n)
3. **Loop that halves each time:** O(log n) -- like binary search
4. **Nested loops both over n:** O(n^2)
5. **Sorting:** O(n log n) minimum
6. **Trying all subsets:** O(2^n) -- avoid at all costs!

---
## Mini-Quiz

In [ ]:
# Q1: What is the Big-O of this code?
def mystery1(data):
    s = set(data)
    return 42 in s
# Answer: O(__)  (building the set is ___, checking membership is ___)

# Q2: What is the Big-O of this code?
def mystery2(data):
    for i in range(len(data)):
        for j in range(10):
            print(data[i])
# Answer: O(__)  (hint: the inner loop is constant, not dependent on n)

# Q3: Your pipeline processes n records. For EACH record, it checks
# if the record's ID exists in a list of 1000 known IDs.
# What is the Big-O? How would you fix it?
# Answer:
# Current: O(__)
# Fixed:   O(__)
# Fix: ___

---
## Homework Preview

This week's homework asks you to:
1. Classify 10 code snippets by Big-O
2. Fix 3 slow functions using the right data structure
3. Time the before/after and prove the speedup
4. Annotate your own project pipeline functions with Big-O

See the homework notebook for full details.

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)